# Thunders AI — Experiments

This notebook runs ablation studies, parameter tuning experiments,
and visualizes comparative results for Thunders AI models.

In [ ]:
# Install Thunders AI if needed
# !pip install thunders-ai[all]

import thunders_ai
from thunders_ai import Trainer, TrainingConfig, Dataset
from thunders_ai.experiments import ExperimentRunner, AblationStudy

print(f"Thunders AI version: {thunders_ai.__version__}")

## 1. Ablation Study

Systematically remove or modify model components to understand their individual contributions.

In [ ]:
# Define the base configuration
base_config = TrainingConfig(
    model_name="thunders-ai/default",
    epochs=3,
    batch_size=32,
    learning_rate=3e-4,
    max_seq_length=512,
    fp16=True,
)

# Define ablation variants
ablation = AblationStudy(
    base_config=base_config,
    variants={
        "no_fp16": TrainingConfig(**{**base_config.__dict__, "fp16": False}),
        "seq_256": TrainingConfig(**{**base_config.__dict__, "max_seq_length": 256}),
        "seq_128": TrainingConfig(**{**base_config.__dict__, "max_seq_length": 128}),
        "lr_1e-4": TrainingConfig(**{**base_config.__dict__, "learning_rate": 1e-4}),
        "lr_1e-3": TrainingConfig(**{**base_config.__dict__, "learning_rate": 1e-3}),
        "batch_16": TrainingConfig(**{**base_config.__dict__, "batch_size": 16}),
        "batch_64": TrainingConfig(**{**base_config.__dict__, "batch_size": 64}),
    },
)

print("Ablation Study Variants:")
for name, config in ablation.variants.items():
    print(f"  {name}: lr={config.learning_rate}, bs={config.batch_size}, seq={config.max_seq_length}, fp16={config.fp16}")

In [ ]:
# Load dataset for ablation
dataset = Dataset.from_huggingface("imdb", split="train")
train_data, val_data = dataset.split(test_size=0.1, seed=42)

# Run ablation study
ablation_results = ablation.run(
    train_dataset=train_data,
    val_dataset=val_data,
    metric="accuracy",
)

# Display results
print("\nAblation Study Results:")
print(f"{'Variant':<15} {'Accuracy':>10} {'Loss':>10} {'Train Time':>12} {'Δ Accuracy':>12}")
print("-" * 62)

baseline_acc = ablation_results["baseline"]["accuracy"]
for name, result in ablation_results.items():
    delta = result["accuracy"] - baseline_acc
    print(f"{name:<15} {result['accuracy']:>9.4f} {result['loss']:>10.4f} {result['train_time_s']:>10.1f}s {delta:>+11.4f}")

## 2. Hyperparameter Tuning

Search for optimal hyperparameters using grid search or random search.

In [ ]:
import itertools
import json

# Define search space
search_space = {
    "learning_rate": [1e-4, 3e-4, 1e-3],
    "batch_size": [16, 32, 64],
    "warmup_steps": [100, 500, 1000],
}

# Generate all combinations (grid search)
keys = list(search_space.keys())
combinations = list(itertools.product(*search_space.values()))

print(f"Grid Search: {len(combinations)} combinations")
for i, combo in enumerate(combinations[:5]):
    params = dict(zip(keys, combo))
    print(f"  {i+1}. {params}")
print(f"  ... and {len(combinations) - 5} more")

In [ ]:
# Run hyperparameter search (using ExperimentRunner)
runner = ExperimentRunner(
    base_config=base_config,
    train_dataset=train_data,
    val_dataset=val_data,
    metric="accuracy",
    direction="maximize",
)

# Run the search
tuning_results = runner.grid_search(
    search_space=search_space,
    epochs=3,
)

# Display top results
print("\nTop 5 Hyperparameter Configurations:")
print(f"{'Rank':<6} {'Accuracy':>10} {'LR':>10} {'Batch':>8} {'Warmup':>8}")
print("-" * 45)
for i, result in enumerate(sorted(tuning_results, key=lambda x: x["accuracy"], reverse=True)[:5]):
    params = result["params"]
    print(f"{i+1:<6} {result['accuracy']:>9.4f} {params['learning_rate']:>10.1e} {params['batch_size']:>8} {params['warmup_steps']:>8}")

best = max(tuning_results, key=lambda x: x["accuracy"])
print(f"\nBest configuration: {best['params']}")
print(f"Best accuracy: {best['accuracy']:.4f}")

## 3. Results Visualization

Visualize the experimental results for comparison and analysis.

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

# --- Ablation Results Bar Chart ---
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# Prepare ablation data
variant_names = list(ablation_results.keys())
accuracies = [ablation_results[v]["accuracy"] for v in variant_names]
train_times = [ablation_results[v]["train_time_s"] for v in variant_names]

# Accuracy comparison
colors = ["#2ecc71" if v == "baseline" else "#3498db" if accuracies[i] >= baseline_acc else "#e74c3c" for i, v in enumerate(variant_names)]
bars = axes[0].barh(variant_names, accuracies, color=colors, edgecolor="white", linewidth=0.5)
axes[0].axvline(x=baseline_acc, color="gray", linestyle="--", alpha=0.7, label=f"Baseline: {baseline_acc:.4f}")
axes[0].set_xlabel("Accuracy")
axes[0].set_title("Ablation Study — Accuracy")
axes[0].legend()
for bar, acc in zip(bars, accuracies):
    axes[0].text(bar.get_width() + 0.001, bar.get_y() + bar.get_height()/2, f"{acc:.4f}", va="center", fontsize=9)

# Training time comparison
axes[1].barh(variant_names, train_times, color="#9b59b6", edgecolor="white", linewidth=0.5)
axes[1].set_xlabel("Training Time (seconds)")
axes[1].set_title("Ablation Study — Training Time")

plt.tight_layout()
plt.savefig("ablation_results.png", dpi=150, bbox_inches="tight")
plt.show()
print("Ablation results saved to ablation_results.png")

In [ ]:
# --- Hyperparameter Heatmap ---
# Create a heatmap of accuracy vs learning_rate and batch_size
lr_values = sorted(search_space["learning_rate"])
bs_values = sorted(search_space["batch_size"])

# Build heatmap data (average over warmup_steps)
heatmap_data = np.zeros((len(lr_values), len(bs_values)))
for result in tuning_results:
    lr_idx = lr_values.index(result["params"]["learning_rate"])
    bs_idx = bs_values.index(result["params"]["batch_size"])
    heatmap_data[lr_idx, bs_idx] = max(heatmap_data[lr_idx, bs_idx], result["accuracy"])

fig, ax = plt.subplots(figsize=(8, 6))
im = ax.imshow(heatmap_data, cmap="YlGn", aspect="auto")
ax.set_xticks(range(len(bs_values)))
ax.set_xticklabels([str(b) for b in bs_values])
ax.set_yticks(range(len(lr_values)))
ax.set_yticklabels([f"{lr:.0e}" for lr in lr_values])
ax.set_xlabel("Batch Size")
ax.set_ylabel("Learning Rate")
ax.set_title("Accuracy Heatmap: Learning Rate vs Batch Size")

# Add text annotations
for i in range(len(lr_values)):
    for j in range(len(bs_values)):
        ax.text(j, i, f"{heatmap_data[i, j]:.4f}", ha="center", va="center", fontsize=10)

plt.colorbar(im, label="Accuracy")
plt.tight_layout()
plt.savefig("hyperparameter_heatmap.png", dpi=150, bbox_inches="tight")
plt.show()
print("Heatmap saved to hyperparameter_heatmap.png")

In [ ]:
# --- Save All Results ---
import json

all_results = {
    "ablation_study": {
        name: {
            "accuracy": result["accuracy"],
            "loss": result["loss"],
            "train_time_s": result["train_time_s"],
        }
        for name, result in ablation_results.items()
    },
    "hyperparameter_tuning": {
        "best_params": best["params"],
        "best_accuracy": best["accuracy"],
        "all_results": [
            {"params": r["params"], "accuracy": r["accuracy"]}
            for r in sorted(tuning_results, key=lambda x: x["accuracy"], reverse=True)
        ],
    },
}

with open("experiment_results.json", "w") as f:
    json.dump(all_results, f, indent=2)

print("All experiment results saved to experiment_results.json")